# Diabetes Model

## Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import mean_squared_error as root_mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

## Cargar el conjunto de datos

In [2]:
diabetes_df = pd.read_excel('../data/raw/M07-M08-M09.xlsx', sheet_name='Diabetes')
diabetes_df.info()
diabetes_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## Limpieza de datos

In [3]:
gender_dict = {'Male': 0, 'Female': 1}
diabetes_df['gender'] = diabetes_df['gender'].map(gender_dict)

print(diabetes_df['gender'].value_counts())

gender
1.0    58552
0.0    41430
Name: count, dtype: int64


In [4]:
# Exploramos los valores de la columna 'smoking_history'
smoking_values = diabetes_df['smoking_history'].value_counts()
print(smoking_values)

smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64


In [5]:
smoking_dict = {'No Info': 0, 'never': 1, 'former': 2, 'current': 3, 'not current': 4, 'ever': 5}
diabetes_df['smoking_history'] = diabetes_df['smoking_history'].map(smoking_dict)

print(diabetes_df['smoking_history'].value_counts())

smoking_history
0    35816
1    35095
2     9352
3     9286
4     6447
5     4004
Name: count, dtype: int64


In [6]:
# Exploramos edades en 8 rangos para ver la distribución de los datos
age_ranges = pd.qcut(diabetes_df['age'], 8).value_counts()

print(age_ranges)

age
(0.079, 13.0]    13305
(51.0, 60.0]     13212
(24.0, 34.0]     12875
(13.0, 24.0]     12580
(70.0, 80.0]     12356
(34.0, 43.0]     12246
(43.0, 51.0]     12150
(60.0, 70.0]     11276
Name: count, dtype: int64


In [7]:
# Categorizamos las edades para que ningun grupo incluya más
# características que otro grupo, y así evitar sesgos en el modelo
bins = [-np.inf, 13, 24, 34, 43, 51, 60, 70, np.inf]
labels = [0, 1, 2, 3, 4, 5, 6, 7]

diabetes_df['age'] = pd.cut(
  pd.to_numeric(diabetes_df['age'], errors='coerce'),
  bins=bins,
  labels=labels
)

diabetes_df['age'] = diabetes_df['age'].astype('Int64')
diabetes_df['age'].value_counts().sort_index()

age
0    13305
1    12580
2    12875
3    12246
4    12150
5    13212
6    11276
7    12356
Name: count, dtype: Int64

In [8]:
# Visualizamos la tabla resultante
diabetes_df.info()
diabetes_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               99982 non-null   float64
 1   age                  100000 non-null  Int64  
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  int64  
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: Int64(1), float64(3), int64(5)
memory usage: 7.0 MB


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,1.0,7,0,1,1,25.19,6.6,140,0
1,1.0,5,0,0,0,27.32,6.6,80,0
2,0.0,2,0,0,1,27.32,5.7,158,0
3,1.0,3,0,0,3,23.45,5.0,155,0
4,0.0,7,1,1,3,20.14,4.8,155,0


## Preparación de los datos

In [9]:
y = diabetes_df['diabetes']
X = diabetes_df.drop('diabetes', axis=1)

# Separación de los datos en subconjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Entrenamiento: {len(X_train)}   Prueba: {len(X_test)}')

Entrenamiento: 80000   Prueba: 20000


## Funciones para evaluar el modelo

In [10]:
# Validación cruzada de 5 pliegues, con la misma semilla para que sea reproducible
cv = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluar(y_real, y_pred):
  """Métricas de clasificación para un vector de predicciones."""
  return {
    'Accuracy': accuracy_score(y_real, y_pred),
    'Precision': precision_score(y_real, y_pred, zero_division=0),
    'Recall': recall_score(y_real, y_pred, zero_division=0),
    'F1': f1_score(y_real, y_pred, zero_division=0),
    'RMSE': root_mean_squared_error(y_real, y_pred),
    'R2': r2_score(y_real, y_pred)
  }

def rmse_cv(modelo, X, y):
  """RMSE promedio de validación cruzada sobre el conjunto de ENTRENAMIENTO."""
  puntajes = cross_val_score(modelo, X, y, cv=cv, scoring='neg_root_mean_squared_error')
  return -puntajes.mean()

resultados = {}

## Entrenamiento del modelo

### Modelo simple (Dummy Classifier)

In [11]:
base = DummyClassifier(strategy='most_frequent')
base.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.92,0.08]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](8,)","['gender','age','hypertension',...,'bmi','HbA1c_level', 'blood_glucose_level']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,8
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [12]:
resultados['Base'] = evaluar(y_test, base.predict(X_test))
print(resultados['Base'])

{'Accuracy': 0.9146, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'RMSE': 0.0854, 'R2': -0.09337415263503179}


### Modelo árbol de decisión (Decision Tree Classifier)

In [13]:
from sklearn.tree import DecisionTreeClassifier


decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

resultados['Decision Tree'] = evaluar(y_test, decision_tree.predict(X_test))
print(resultados['Decision Tree'])

{'Accuracy': 0.9536, 'Precision': 0.7238805970149254, 'Recall': 0.7382903981264637, 'F1': 0.7310144927536232, 'RMSE': 0.0464, 'R2': 0.40594191238565014}


### Modelo Estocast (Gradient Boosting Classifier)

In [14]:
from sklearn.linear_model import SGDClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sgd = make_pipeline(
  SimpleImputer(strategy='median'),
  StandardScaler(),
  SGDClassifier(max_iter=5, tol=None, random_state=42)
)

sgd.fit(X_train, y_train)

resultados['SGD'] = evaluar(y_test, sgd.predict(X_test))
print(resultados['SGD'])

{'Accuracy': 0.9552, 'Precision': 0.8949416342412452, 'Recall': 0.5386416861826698, 'F1': 0.672514619883041, 'RMSE': 0.0448, 'R2': 0.4264266740275242}


### Modelo de regresión logística (Logistic Regression)

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic = make_pipeline(
  SimpleImputer(strategy='median'),
  StandardScaler(),
  LogisticRegression(max_iter=1000, random_state=42)
)

logistic.fit(X_train, y_train)

resultados['Logistic Regression'] = evaluar(y_test, logistic.predict(X_test))
print(resultados['Logistic Regression'])

{'Accuracy': 0.959, 'Precision': 0.8712374581939799, 'Recall': 0.6100702576112412, 'F1': 0.7176308539944903, 'RMSE': 0.041, 'R2': 0.47507798292697534}


### Modelo Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

random_forest = make_pipeline(
  SimpleImputer(strategy='median'),
  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
)

random_forest.fit(X_train, y_train)

resultados['Random Forest'] = evaluar(y_test, random_forest.predict(X_test))
print(resultados['Random Forest'])

{'Accuracy': 0.96965, 'Precision': 0.9317647058823529, 'Recall': 0.6955503512880562, 'F1': 0.7965135769359705, 'RMSE': 0.03035, 'R2': 0.6114296776057}


## Comparación de modelos

Se ordenan los modelos por RMSE y R2 para comparar su desempeño (diabetes = 1).

In [17]:
comparacion = pd.DataFrame(resultados).T.sort_values('RMSE', ascending=True)
comparacion.round(4)

,Accuracy,Precision,Recall,F1,RMSE,R2
Random Forest,0.9696,0.9318,0.6956,0.7965,0.0304,0.6114
Logistic Regression,0.9590,0.8712,0.6101,0.7176,0.0410,0.4751
SGD,0.9552,0.8949,0.5386,0.6725,0.0448,0.4264
Decision Tree,0.9536,0.7239,0.7383,0.7310,0.0464,0.4059
Base,0.9146,0.0000,0.0000,0.0000,0.0854,-0.0934


## Exportar el modelo final

In [18]:
import joblib

# Exportamos el modelo con mejor desempeño
joblib.dump(random_forest, '../models/random_forest_model.joblib')

['../models/random_forest_model.joblib']